In [66]:
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

In [5]:
gdf_2022 = gpd.read_file('2022data.shp')
gdf_2022 = gdf_2022.to_crs("32718")

In [60]:
df_water_2024 = pd.read_excel('2024_full_data.xls')

In [61]:
df_water_2024.columns

Index(['OBJECTID', 'Join_Count', 'TARGET_FID', 'Speed_km_h',
       'Direction_of_travel', 'Compass_reading', 'Position_source_type',
       'Receiver_Name', 'Horizontal_Accuracy_m', 'Vertical_Accuracy_m',
       'Latitude', 'Longitude', 'Altitude', 'PDOP', 'HDOP', 'VDOP', 'Fix_Type',
       'Correction_Age', 'Station_ID', 'Number_of_Satellites', 'Fix_Time',
       'Average_Horizontal_Accuracy_m', 'Average_Vertical_Accuracy_m',
       'Averaged_Positions', 'Standard_Deviation_m', 'CreationDate', 'Creator',
       'EditDate', 'Editor', 'Additional_observations', 'Land_use_type',
       'Land_use_description', 'Underlying_geological_feature',
       'Underlying_geomorphological_feature_legend_code',
       'Complete_soil_description', 'Soil_code_abbreviation', 'Soil_type',
       'Altitude_1', 'Slope_angle', 'Data_point_number', 'Date_and_time',
       'AREA', 'PERIMETER', 'GEOLOGIA', 'GEOLOGIA_I', 'CODI', 'NAME', 'HPI',
       'CNA', 'VER', 'CAL', 'CA1', 'CON', 'HP', 'CAR', 'MAP', 'CMY'

There are X and Y coordinates for the location of points, I'm gonna use that to make a geodataframe. I don't have access to any kind of
.gpkg or .shp file for the 2024 data so relying on this CSV will have to do

In [7]:
gdf_2024 = gpd.GeoDataFrame(
    df_water_2024,
    geometry=gpd.points_from_xy(df_water_2024["x"], df_water_2024["y"]),
    crs='EPSG:32718',
)

In [44]:
gdf_2026 = gpd.read_file('2026_water_samples_geo.gpkg')

/Users/Sari/Documents/Code/watergroup/venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: Non-conformant content for record 1 in column CreationDate, 2026-06-14T17:25:28.0Z, successfully parsed
  return ogr_read(


In [45]:
lab_results_2026 = pd.read_csv('2026_lab_results.csv')

In [47]:
gdf_2026 = gdf_2026.merge(
    lab_results_2026,
    left_on=('ID'),
    right_on=('Sample'),
)

In [51]:
gdf_2026 = gdf_2026.drop(columns='Sample')

In [48]:
# simplifying some column names
gdf_2026 = gdf_2026.rename(columns={
    "Tree_cover": "tree_cover", 
    "Shrubs_cover": "shrub_cover", 
    "SucculentsCacti_cover": "succulent_cover",
    "Grasses_cover": "grass_cover",
    "Herbaceous_plant_cover": "herbaceous_cover",
    "Bare_cover": "bare_cover",
    "Electrical_Conductivity": "EC",
    'Al\n(mg/L)': 'Al',
    'As\n(mg/L)': 'As',
    'Ba\n(mg/L)': 'Ba',
    'Ca\n(mg/L)': 'Ca',
    'Cd\n(mg/L)': 'Cd',
    'Cu\n(mg/L)': 'Cu',
    'Fe\n(mg/L)': 'Fe',
    'K\n(mg/L)': 'K',
    'Mg\n(mg/L)': 'Mg',
    'Na\n(mg/L)': 'Na',
    'Pb\n(mg/L)': 'Pb',
    'S\n(mg/L)': 'S',
})

In [11]:
element_cols = ['Al', 'As', 'Ba', 'Ca', 'Cd', 'Cu', 'Fe', 'K', 'Mg', 'Na', 'Pb', 'S']
vegetation_cover_columns = ['tree_cover', 'shrub_cover', 'succulent_cover', 'grass_cover', 'herbaceous_cover', 'bare_cover']

In [12]:
# so we entered the data in a little bit of an insane way.
# For some samples, we did observe vegetation surrounding the sample. For these, if a category had 0% cover, we either entered a 0 value,
# or no value at all (a NaN).
# for the samples where we didn't collect anything, every category for vegetation cover is a NaN
# I can't just make all NaNs 0, because it would be wrong: we just don't have data for some of these, so they aren't 0. They would
# mess up any model we might make. but I also need to replace the NaNs with 0s on the data points we DID collect but didn't input a 0 in

In [18]:
# checks if any of the veg cover values isn't a NaN across each row
measured_vegetation = gdf_2026[vegetation_cover_columns].notna().any(axis=1)
gdf_2026['measured_vegetation'] = measured_vegetation
gdf_2026.loc[measured_vegetation, vegetation_cover_columns] = (
    gdf_2026.loc[measured_vegetation, vegetation_cover_columns].fillna(0)
)

In [19]:
df_measured_vegetation = gdf_2026[measured_vegetation].copy()

In [20]:
df_measured_vegetation[vegetation_cover_columns].sum(axis=1) - df_measured_vegetation['bare_cover']

0      58.0
2     109.0
5      43.0
6      64.0
16    115.0
19     86.0
22     93.0
61    141.0
63     95.0
72     44.0
73     80.0
76     94.0
dtype: float64

In [17]:
measured_vegetation = gdf_2026[vegetation_cover_columns].notna().any(axis=1)
gdf_2026.index[measured_vegetation].tolist()

[0, 2, 5, 6, 16, 19, 22, 61, 63, 72, 73, 76]

In [23]:
sorted(gdf_2026.loc[measured_vegetation, 'ID'].tolist(), key=int)

['101',
 '102',
 '105',
 '108',
 '110',
 '111',
 '112',
 '116',
 '124',
 '178',
 '182',
 '183']

In [68]:
groups = ['river', 'river', 'river', 'river', 'lake', 'lake', 'lake',]

In [67]:
years = [2024, 2024, 2026, 2026, 2024, 2024, 2026]

In [96]:
ph = [7, 7, 8, 8, 6, 7, 8]
id = ['010', '011', '12', '13', '014', '017', '20',]

In [97]:
df_all_years_by_area = pd.DataFrame({
    'area': groups,
     'year': years, 
     'ph': ph,
     'ID': id,
})

In [ ]:
string()

In [101]:
df_all_years_by_area['label'] = df_all_years_by_area['year'].astype(str) + df_all_years_by_area['ID']

In [102]:
df_all_years_by_area['label']

0    2024010
1    2024011
2     202612
3     202613
4    2024014
5    2024017
6     202620
Name: label, dtype: str

In [113]:
gdf_2022['label'] = '2022' + gdf_2022['Obs_number']
gdf_2024['label'] = '2024' + gdf_2024['OBJECTID'].astype(str)
gdf_2026['label'] = '2026' + gdf_2026['ID']

In [85]:
gdf_2026['low_cover'] = (
    gdf_2026['herbaceous_cover'] 
    + gdf_2026['grass_cover']
    + gdf_2026['succulent_cover']
)

In [86]:
gdf_2026['high_cover'] = (
    gdf_2026['tree_cover']
    + gdf_2026['shrub_cover']
)

In [ ]:
df_all_years_by_area.merge(
    gdf_2026,
    on='label'
)



TypeError: DataFrame.merge() missing 1 required positional argument: 'right'

'055'